its not really working. im going to halt work here, and simply write the article on our work.  
Its most likely very similar to FastICA. But oh well.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import cauchy

x = np.linspace(-2, 2, 1000)

params = [
    (0, 0.5),
    (0, 1.0),
    (0, 2.0),
    (0, 0.001),
]

fig, axes = plt.subplots(1, 1, figsize=(14, 5))

for x0, gamma in params:
    pdf = cauchy.pdf(x, loc=x0, scale=gamma)
    cdf = cauchy.cdf(x, loc=x0, scale=gamma)
    label = f"x₀={x0}, γ={gamma}"
    axes.plot(x, pdf, label=label)

axes.set_title("Cauchy PDF")
axes.set_xlabel("x")
axes.set_ylabel("f(x)")
axes.set_ylim(0, 0.8)
axes.legend(fontsize=8)
axes.grid(True, alpha=0.3)


plt.suptitle("Cauchy Distribution", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
n, d, k = 500, 10, 3

x0 = 0

X = np.random.randn(n, d)
target_gamma = 0.001
gamma = target_gamma / np.mean(np.abs(X).sum(axis=1))
gamma


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import cauchy


W = cauchy.rvs(loc=x0, scale=gamma, size=(k, d))
b = cauchy.rvs(loc=x0, scale=gamma, size=(k,))

# linear forward pass: (n, d) @ (d, k) + (k,) -> (n, k)
out = X @ W.T + b  # (n, k)

fig, axes = plt.subplots(1, k, figsize=(4 * k, 4), sharey=False)

for i, ax in enumerate(axes):
    ax.hist(out[:, i], bins=50, density=True, alpha=0.7)
    ax.set_title(f"output dim {i}")
    ax.set_xlim(-1, 1)
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Output distribution  (n={n}, d={d}, k={k}, x₀={x0}, γ={gamma})", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
x0_fit, gamma_fit = cauchy.fit(out)
gamma_fit


# Start topic modelling

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools
from torch import nn
from torch import optim


In [ ]:
from torch import nn
from torch.nn import functional as F
from torch import optim


class SparseAE(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=False),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        # recon = F.softplus(self.decoder(codes))
        recon = self.decoder(codes)
        return recon, codes

    def poisson_recon_loss(self, x, recons):
        # recons should be passed through softplus before this
        return torch.nn.functional.poisson_nll_loss(
            recons, x, log_input=False, full=False, reduction='mean'
        )

    def loss_fn(self, x, recons, codes, alpha, sigma_x, sigma_s, sigma_0):
        recons_loss = self.recon_loss(x, recons, sigma_x)

        codes_loss = self.codes_loss(codes, sigma_s)

        weights_loss = self.weights_loss(alpha, sigma_0)

        return recons_loss, codes_loss, weights_loss

    def codes_loss(self, codes, sigma_s):
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def recon_loss(self, x, recons, sigma_x):
        return self.gauss_loss(x, recons) / (sigma_x * sigma_x)

    def cauchy_loss(self, x, gamma):
        # return torch.log(1 + (x * x) / (target_gamma_on_codes * target_gamma_on_codes)).sum()
        # gamma = target_gamma_on_codes 
        return torch.log(gamma + (x*x)/(gamma*gamma)).sum(1).mean()

    def cauchy_codes_loss(self, codes, cauchy_gamma):
        return (
            torch.log(1 + (codes * codes) / (cauchy_gamma * cauchy_gamma)).sum(1).mean()
        )

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        # sum the loss inside one example, send back mean across examples for the batch
        return torch.sum(loss, 1).mean()

    def l1_loss(self, x, mean):
        loss = (x - mean).abs()
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0):
        W = self.decoder.weight
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

    def poisson_loss(self, x, recon):
        # recon should be non-negative — apply softplus or relu before passing in
        eps = 1e-8
        loss = recon - x * torch.log(recon + eps)
        return torch.sum(loss, 1).mean()

    def masked_recon_loss(self, x, recons, sigma_x):
        mask = (x != 0).float()
        loss = ((x - recons) ** 2) * mask
        return torch.sum(loss, 1).mean() / (sigma_x**2)

    def weights_loss_cycled(self, alpha, sigma_0, chunk_size=64):
        W = self.decoder.weight          # (C, K)
        C, K = W.shape

        idx = (torch.arange(K, device=W.device).unsqueeze(0) +
            torch.arange(K, device=W.device).unsqueeze(1)) % K   # (K, K)

        shift_losses = []

        for start in range(0, K, chunk_size):
            idx_chunk = idx[start:start + chunk_size]   # (chunk, K)

            W_chunk = W[:, idx_chunk]                   # (C, chunk, K)
            W_chunk = W_chunk.permute(1, 0, 2)          # (chunk, C, K)

            W_sq = W_chunk ** 2

            cumsum = torch.cumsum(W_sq, dim=2)
            phi = alpha * torch.roll(cumsum, 1, dims=2) + 1
            phi[:, :, 0] = 1

            comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
            comp2 = -torch.log(phi)

            shift_losses.append((comp1 + comp2).sum(dim=(1, 2)))  # (chunk,)

        return torch.cat(shift_losses).mean()

def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)


def train_grouping_autoencoder_fixed_sigma(
    X,
    n_components,
    max_alpha=5000,
    sigma_x=0.1,
    sigma_s=0.1,
    codes_l1_b=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_x: std of noise in the data after modelling it as a WS
    sigma_0: std of W, useful to keepn very near 0
    sigma_s: cauchy gamma for cauchy penalty on the encoder (useful for sparse weights)
       the name is sigma_s, cuz it was used as a gaussian prior (L2) on encoder weights, for data which is not sparse
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = GroupingAutoencoderForSparseData(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)

    # svd, might add back again later
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]

            # first do on recons, let everything flow, make LR for weight 0
            recon, codes = model(batch)
            recon_loss = model.poisson_recon_loss(batch, recon)
            alpha = get_alpha(epoch, epochs, 100, max_alpha)
            if weights_algo == "cycled":
                weight_loss = model.weights_loss_cycled(alpha, sigma_0)
            else:
                comp1, comp2 = model.weights_loss(alpha, sigma_0)
                weight_loss = (comp1 + comp2).sum()
            

            # # code_loss = model.cauchy_codes_loss(model.encoder[0].weight, cauchy_gamma)
            # maker_loss = model.gauss_loss(model.codes_maker.weight, 0) / (
            #     sigma_s * sigma_s
            # )
            maker_loss = -1
            code_loss = model.l1_loss(codes, 0) / codes_l1_b

            # if epoch > 5:
            #     loss = recon_loss + code_loss + weight_loss + maker_loss
            # else:
            loss = recon_loss + code_loss + weight_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            msg = f"epoch {epoch:4d} | recon_loss {recon_loss:.4f}"
            if weight_loss is not None:
                # msg += f" weight_comp1 {weights_comp1.sum():.4f} weights_log {weights_log_comp.sum():.4f} codes_loss {code_loss:.4f}"
                msg += f" weight_loss {weight_loss.sum():.4f} codes_loss {code_loss:.4f} maker_loss {maker_loss:.4f}" 
            # msg += f" reg_weight: {reg_weight}"
            print(msg)

    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        (codes).numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def train_baseline(X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, l1_b=1):
    # simple linear model without any non linearities
    # for loss checking
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = SparseAE(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            recon_loss = model.recon_loss(batch, recon, sigma_x)
            sparse_weight_loss = model.l1_loss(codes, 0) / l1_b

            loss = recon_loss + sparse_weight_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f} sparse_weight_loss {sparse_weight_loss}")

    with torch.no_grad():
        recon, codes, _ = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def show_gram(W):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
    gram = W_norm.T @ W_norm  # (n_components, n_components)
    plt.imshow(gram, cmap="gray")
    plt.show()
    return gram

# Topic model

In [ ]:
from time import time

import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF, LatentDirichletAllocation, MiniBatchNMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

n_samples = 2000
n_features = 1000
n_components = 10
n_top_words = 20
batch_size = 128
init = "nndsvda"


def plot_top_words(components_, feature_names, n_top_words, title):
    fig, axes = plt.subplots(2, 5, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()


# Load the 20 newsgroups dataset and vectorize it. We use a few heuristics
# to filter out useless terms early on: the posts are stripped of headers,
# footers and quoted replies, and common English words, words occurring in
# only one document or in at least 95% of the documents are removed.

print("Loading dataset...")
t0 = time()
data, _ = fetch_20newsgroups(
    shuffle=True,
    random_state=1,
    remove=("headers", "footers", "quotes"),
    return_X_y=True,
)
data_samples = data[:n_samples]
print("done in %0.3fs." % (time() - t0))

# Use tf-idf features for NMF.
print("Extracting tf-idf features for NMF...")
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95, min_df=2, max_features=n_features, stop_words="english"
)
t0 = time()
tfidf = tfidf_vectorizer.fit_transform(data_samples)
print("done in %0.3fs." % (time() - t0))


In [ ]:
print(data_samples[0])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
ns = []
for i, v in enumerate(tfidf.toarray()[0]):
    if v == 0:
        continue
    ns.append(tfidf_feature_names[i])
print(ns)

## NMF


In [ ]:
# Fit the NMF model
print(
    "Fitting the NMF model (Frobenius norm) with tf-idf features, "
    "n_samples=%d and n_features=%d..." % (n_samples, n_features)
)
t0 = time()
nmf = NMF(
    n_components=n_components,
    random_state=1,
    init=init,
    beta_loss="frobenius",
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=1,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    nmf.components_, tfidf_feature_names, n_top_words, "Topics in NMF model (Frobenius norm)"
)



## SAE

In [ ]:
tfarr = tfidf.toarray()

In [ ]:
model, codes, components, recon = train_baseline(tfarr, n_components, 1e-3, 3000)

noise = tfarr - recon
# mean is very close to zero
print("overall stats", noise.mean(), noise.std())
plt.plot(noise.std(0))
plt.show()

In [ ]:
d = DictionaryLearning(10, fit_algorithm="cd", transform_algorithm="lasso_cd")
d.fit(tfarr)

codes = d.transform(tfarr)
(codes @ d.components_).shape, tfarr.shape

error = np.mean((tfarr - (codes @ d.components_)) ** 2)

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    d.components_, tfidf_feature_names, n_top_words, "Topics in dict learning lasso_cd"
)

In [ ]:
d = DictionaryLearning(10, fit_algorithm="lars", transform_algorithm="lasso_lars")
d.fit(tfarr)

codes = d.transform(tfarr)
(codes @ d.components_).shape, tfarr.shape

error = np.mean((tfarr - (codes @ d.components_)) ** 2)

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    d.components_, tfidf_feature_names, n_top_words, "Topics in dict learning lasso_lars"
)

In [ ]:
error

In [ ]:
recons = 

In [ ]:
np.max(recon[0])

In [ ]:
X.std()

In [ ]:
noise_std = 0.01
sigma_x = noise_std
sigma_w = 1
sigma_s = 1
codes_l1_b = 1
print( "w", sigma_w, "noise", noise_std)

model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    tfarr, 
    n_components, 
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_0=sigma_w,
    codes_l1_b=codes_l1_b,
    sigma_s=sigma_s,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    components, tfidf_feature_names, n_top_words, "Topics in NMF model (Frobenius norm)"
)
